# 00 — Setup and index

Spin Qdrant, ingest scifact, sanity-check the collection. Run this once before the rest of the tour.


In [ ]:
from rag_evals.config import settings
from rag_evals.index.qdrant_store import QdrantStore

store = QdrantStore()
store.ensure_collection()
print(f"collection={store.collection!r} url={store.url!r}")


Run `make index` from the shell to ingest scifact and build golden sets — that step is heavy
(downloads embeddings + reranker the first time), so we do it outside the notebook.

Below: verify what landed in the collection.


In [ ]:
from rag_evals.data import scifact
from rag_evals.data.metadata import synthesize

n = sum(1 for _ in scifact.documents())
print(f"scifact corpus size: {n}")

# Show metadata distribution on a sample
sample = list(scifact.documents())[:200]
from collections import Counter
for field in ("tenant", "locale", "domain"):
    counts = Counter(synthesize(d.doc_id)[field] for d in sample)
    print(f"  {field:>7}: {dict(counts)}")


In [ ]:
# Smoke retrieval if the index has been seeded
try:
    count = store.count()
    print(f"qdrant collection has {count} points")
except Exception as e:
    print(f"collection empty or unreachable: {e}")


In [ ]:
store.close()
